# Melakukan inferensi terhadap model yang telah di deploy

In [1]:
import tensorflow as tf
import base64
import requests

2026-05-18 15:32:48.721058: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-18 15:32:49.158291: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-05-18 15:32:49.158323: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-05-18 15:32:49.209666: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-18 15:32:50.226913: W tensorflow/stream_executor/platform/de

In [2]:
def process_text_fn(review_text):

    feature = {
        "review": tf.train.Feature(bytes_list=tf.train.BytesList(value=[review_text.encode('utf-8')]))
    }
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    serialized_example = example.SerializeToString()
    b64_encoded_example = base64.b64encode(serialized_example).decode('utf-8')
    payload = {
        "instances": [
            {"b64": b64_encoded_example}
        ]
    }

    return payload

In [5]:

reviews = [
    "The new skill tree mechanics are incredibly deep and rewarding",
    "optimization is terrible i always get stutters",
    "10/10",
    "1/10",
    "the anti cheat is good",
    "this game company is so greedy",
    "game is good, but its not for me"
]

for i, review in enumerate(reviews,1):
    payload = process_text_fn(review_text=review)
    try:
        TF_SERVING_URL = "https://izzan-virm-review-classification-production.up.railway.app//v1/models/review-classification-model:predict"
        
        response = requests.post(TF_SERVING_URL, json=payload)
        response.raise_for_status()
        
        print(f"\nReview text: {review}")

        result = response.json()
        prediction = result['predictions'][0][0]
        
        print(f"Extracted Sentiment Score: {prediction:.4f}")
        if prediction > 0.5:
            print("Result: Recommended")
        else:
            print("Result: Not Recommended")

    except requests.exceptions.ConnectionError:
        print("\n[Error] No Connection")
    except Exception as e:
        print(f"\n[Error]: {str(e)}")


Review text: The new skill tree mechanics are incredibly deep and rewarding
Extracted Sentiment Score: 0.9602
Result: Recommended

Review text: optimization is terrible i always get stutters
Extracted Sentiment Score: 0.0405
Result: Not Recommended

Review text: 10/10
Extracted Sentiment Score: 0.9799
Result: Recommended

Review text: 1/10
Extracted Sentiment Score: 0.0696
Result: Not Recommended

Review text: the anti cheat is good
Extracted Sentiment Score: 0.7444
Result: Recommended

Review text: this game company is so greedy
Extracted Sentiment Score: 0.0276
Result: Not Recommended

Review text: game is good, but its not for me
Extracted Sentiment Score: 0.8116
Result: Recommended
